In [1]:
!uv pip install langchain langchain-community langchain-core langchain-chroma sentence-transformers pypdf ipywidgets

Using Python 3.13.5 environment at: /home/stangler/Documents/Python/EYE.AI/.venv
Audited 5 packages in 174ms


In [2]:
import os
import json
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

/home/stangler/Documents/Python/EYE.AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'langchain.prompts'

In [ ]:


# ==========================================
# CONFIGURAÇÕES GERAIS
# ==========================================
VECTOR_DB_PATH = "./chroma_db_hermes"
# Ajuste para a tag do modelo que você tem no Ollama (ex: "gemma:2b", "gemma:7b")
MODEL_NAME = "gemma" 

# ==========================================
# 1. INGESTÃO RAG (Knowledge Base Controlada)
# ==========================================
def pipeline_ingestao(caminho_arquivo: str) -> Chroma:
    print(f"\n[1] Iniciando ingestão do arquivo: {caminho_arquivo}")
    
    # Carregamento do documento (Usando TXT para o MVP, substituível por PyPDFLoader)
    loader = TextLoader(caminho_arquivo, encoding="utf-8")
    documentos = loader.load()

    # Quebra em chunks otimizados para captura de contexto médico
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_documents(documentos)

    # Uso de embeddings locais (leves e eficientes para CPU/GPU)
    print(f"[1] Gerando embeddings para {len(chunks)} chunks...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Armazenamento vetorial persistente
    db = Chroma.from_documents(chunks, embeddings, persist_directory=VECTOR_DB_PATH)
    print("[1] Ingestão concluída com sucesso.")
    
    return db

# ==========================================
# 2a. LLM INFERÊNCIA COM RAG
# ==========================================
def pipeline_inferencia_rag(pergunta: str, db: Chroma) -> str:
    print(f"\n[2a] Buscando literatura científica padrão-ouro para: '{pergunta}'")
    
    # Recuperação dos 3 chunks mais relevantes
    retriever = db.as_retriever(search_kwargs={"k": 3})
    docs_relevantes = retriever.invoke(pergunta)
    contexto = "\n\n".join([doc.page_content for doc in docs_relevantes])

    print("[2a] Contexto recuperado. Gerando inferência com modelo local...")
    
    # Instância do Ollama forçando saída estruturada
    llm = Ollama(model=MODEL_NAME, format="json")

    # Prompt rigoroso para evitar alucinação e impor o formato JSON
    template = """Você é o Visio-Chat Hermes, um ChatBot Preceptor especialista em suporte à decisão clínica.
    Sua tarefa é analisar o contexto fornecido (literatura científica e protocolos institucionais) e responder à pergunta do médico.
    
    REGRA DE OURO: Baseie-se EXCLUSIVAMENTE no contexto. Se a resposta não estiver no contexto, declare isso. Nunca invente dados.

    CONTEXTO INSTITUCIONAL:
    {contexto}

    PERGUNTA DO MÉDICO:
    {pergunta}

    Sua resposta DEVE ser um objeto JSON perfeitamente formatado contendo EXATAMENTE estas chaves:
    - "decisao_clinica": (string) A conduta ou resposta baseada nos protocolos.
    - "justificativa": (string) Breve explicação extraída do texto.
    - "alerta_alucinacao": (booleano) false se encontrou a resposta no contexto, true se a informação não estava lá.

    Retorne APENAS o JSON e nada mais.
    """
    
    prompt = PromptTemplate.from_template(template)
    chain = prompt | llm
    
    # Execução da chain
    resposta_bruta = chain.invoke({"contexto": contexto, "pergunta": pergunta})
    return resposta_bruta

# ==========================================
# 2b. OUTPUT JSON ESTRUTURADO (Para MCP)
# ==========================================
def formatar_output_json(resposta_bruta: str) -> str:
    print("\n[2b] Validando e estruturando output para o MCP...")
    try:
        # Tenta fazer o parse para garantir que o LLM não quebrou o contrato do JSON
        dados = json.loads(resposta_bruta)
        return json.dumps(dados, indent=4, ensure_ascii=False)
    except json.JSONDecodeError:
        # Fallback de segurança caso o modelo local falhe na formatação
        print("Aviso: O modelo não retornou um JSON estrito. Tentativa de recuperação aplicada.")
        return json.dumps({
            "erro_parse": True,
            "conteudo_bruto": resposta_bruta
        }, indent=4, ensure_ascii=False)

# ==========================================
# EXECUÇÃO DO MVP
# ==========================================
if __name__ == "__main__":
    # Setup de simulação: Criando um arquivo de literatura controlado [cite: 44, 45]
    arquivo_teste = "protocolo_teste.txt"
    with open(arquivo_teste, "w", encoding="utf-8") as f:
        f.write("Protocolo Institucional de Cardiologia (v2.1): Em pacientes adultos com suspeita clínica "
                "de Infarto Agudo do Miocárdio (IAM), o primeiro passo imediato é a administração de "
                "Ácido Acetilsalicílico (AAS) 300mg mastigável, exceto em pacientes com histórico documentado "
                "de anafilaxia ao fármaco ou sangramento ativo severo.")

    # Etapa 1
    vetor_db = pipeline_ingestao(arquivo_teste)

    # Etapa 2a
    pergunta_teste = "Qual o protocolo inicial para um paciente adulto com suspeita de IAM que não tem alergias?"
    output_llm = pipeline_inferencia_rag(pergunta_teste, vetor_db)

    # Etapa 2b
    output_final = formatar_output_json(output_llm)

    print("\n=== PAYLOAD MCP GERADO ===")
    print(output_final)